# Day 20: Monitoring (see what the running service is doing without logging what people asked.)

My API now runs in a container. Today it gets eyes: 
- **one log line per request.**. It says who asked, which route, the status, how long it took, tokens, cost ans whoch error class. It never says what was asked.
- **A `/metrics` page of numbers**, locked behind it's own key.
- **Alert rules** that turn those numbers into "a human is needed" messages.
- **A real-question proof** that monitoring sees a question and leaks nothing.
- **A fix for the memory leaks.** my Day 18 audit found (the lists that only ever grow).

`day20_tools.zip` it attached as a file.

## What I tested, and what I did not
Tested (all passed on the final files):

| What | Result |
|------|--------|
| Six Day 20 test suites(real web server, real HTTP, real log and metrics) | 101 test pass |
| Day 18 suites (39) and Day 19 suite (38), re-run against the changed `serve.py`, `make_container_files.py`, `make_requirements.py` | all still pass |
| Deliberate bugs: I broke the tools 23 ways (secret filter off, token double-counting, open `/metrics`, and so on) | all 23 were caught by the tests |
| Whole day in one real Jupyter kernel: your Day 19 cells, then the 9 cells below, with a real Docker build and run | 0 cell errors |
| 12 tests against real running containers (list below) | 12/12 pass |

### **The container tests covered:**
- Log lines are JSON only, with no secrets.
- A timeout followed by a late finish shows up correctly.
- A restart is noticed.
- A grateful stop logs a shutdown event.
- 1,500 requests keep memory flat (59 MB to 59 MB) and cap the list at 1,000.
- A leak through `print` is caught when I turn prints on.


### **Three Different Questions, Three tools.**
| Question | Tool | Example |
|----------|------|---------|
| What happened to this one request? | Log | "request 5fc6... took 413 ms, 1,952 tokens in" |
| How is the whole service doing over time? | **Metrics**(numbers that only go up, or go up and down) | "1,204 questions answered; 3 failed" |
| Does a human need to look right now? | **Alert**(a rule over the numbers) | "Redis is not answering" |

**Why one JSON object per line.** A sentence like `rid=abc POST /ask -> 200 413ms` is easy to read but hard to search. `{"rid":"abc","route":"/ask","status": 200, "duration_ms":413}` can be filtered, summmed and averaged by a program (`log_report.py` does exactly that).

**What must never be logged.** Logs get copied, sent to other tools and kept for months. So they hold facts about requests, never content: 
- No question text and no answer text.
- No API keys.
- No exception messages, because they often contain coneection strings with passwords. Only the error class and the file and line are logged.
- Session ids are replaced by a short hash. I can still follow one conversation, but nobody can read the id.

### **Three defenses against leaks.**
1. The code never puts content in the log.
2. A filter blanks every secret value found inn my environment(`[redacted]`) in every log line.
3. `print()` calls inside my pipeline are silenced. Notebook code is full of `print(f"Question: {q}")`, and in a container, these will land in the same log.

### Cell 6 will Then prove it with a real question
### **Counters, gauges, histograms.**
- A **counter** only goes up (questions answered, tokens used). I read it by how much it grew in a time window.
- A **gauge** goes up and down (calls running now, memory held).
- A **histogram** counts how many answers took <=1s, <=2.5s, <=5s and so on. It lets me say "95% of answers came within x seconds" (p95). The average hides the slow ones; p95 shows them.
- **Cardinality**. Every different label value(route, status) creates a new number to store. If scanners hit 10,000 random URLs, I would store 10,000 series, So labels come from a small fixed set. Anything unknown becomes `other`, and unknown URLs become `unmatched`.

**The timed-out call that keeps running.** A python thread cannot be killed. When a question takes longer than the timeout, the caller gets 504 but the thread keeps working and keeps spending tokens. Day 17 kept the slot taken until it really ends. Day 20 makes that visible:
- The request line says `pipeline_still_running: true`.
- A later `pipeline_late_finish` line says how it really ended and what extra it cost.
- A counter, `rag_pipeline_abandoned_total`, records it.

### **Why the request record needs special handling.**
 When a request starts, a record is created ("this is request 5fc6..."). My pipeline runs in a worker thread, and `run_in_executor` does not carry that record into the thread. A `ContextExecutor` copies it, so the tokens counted inside the thread land on the right request even with two requests at once. A test with three overlapping quests checks this.

**Alerts compare two looks.** "3 errors" means nothing; "3 errors in the last 5 minutes" does. `alert_rules.py` reads `/metrics` every 30 seconds, compares with what it saw up to 5 minutes ago, and prints a line only when something starts or stops firing.

**Bounded containers.** A list thet every request adds to and nothing removes from is s slow memory leak. `make_bounded.py` swaps each one for a version that drops the oldest entry when full. Only thr line that creates the container changes.






In [1]:
# starting with checking the folder

import os, sys, subprocess, time, json, pathlib, re, threading
sys.path.insert(0, os.getcwd())

# --- 1. The Day 20 files must be in this folder (they replace three older ones: serve.py, make_container_files.py, make_requirements.py)
FILES = ["monitoring.py", "bounded.py", "make_bounded.py", "prepare_monitoring.py", "alert_rules.py", "log_report.py",
         "check_monitoring.py", "testkit.py", "test_monitoring.py", "test_bounded.py", "test_alerts.py", "test_log_report.py",
         "test_check_monitoring.py", "test_prepare.py", "serve.py", "make_container_files.py", "make_requirements.py",
         "api_layer.py", "audit_globals.py", "freeze_kernel.py", "container_check.py", "usage_context.py"]       # the last five came from Days 17-19
absent = [f for f in FILES if not os.path.exists(f)]
assert not absent, f"❌ Missing from {os.getcwd()}: {absent}  (extract day20_tools.zip into this folder; choose 'Replace' when asked)"
assert "_add_monitoring" in pathlib.Path("serve.py").read_text(encoding="utf-8"), "❌ serve.py is still the old one: extract day20_tools.zip again and choose Replace"
assert "monitoring.py" in pathlib.Path("make_container_files.py").read_text(encoding="utf-8"), "❌ make_container_files.py is still the Day 19 one: extract day20_tools.zip again and choose Replace"
for must in ("rag_pipeline/pipeline.py", "rag_pipeline/_parity_expected.json", ".env", "Dockerfile", "docker-compose.yml"):
    assert os.path.exists(must), f"❌ {must} is missing. It comes from Day 18 (freeze, .env) / Day 19 (Docker files)."

# --- 1b. If this notebook was already open on Day 19, Python still remembers the OLD versions of these tools. Forget them.
import importlib
for name in [f[:-3] for f in FILES if not f.startswith("test")] + ["container_patch", "docker_healthcheck"]:
    sys.modules.pop(name, None)
importlib.invalidate_caches()

# --- 2. Packages the tools need
for mod, pkg in (("httpx", "httpx"), ("dotenv", "python-dotenv"), ("uvicorn", "uvicorn"), ("fastapi", "fastapi"), ("yaml", "pyyaml")):
    try:
        __import__(mod)
    except ImportError:
        raise AssertionError(f"❌ {pkg} is not installed. Run:  %pip install {pkg}")

# --- 3. A helper that runs docker commands and gives back (exit code, text)
def dc(*args, timeout=1800):
    try:
        r = subprocess.run(["docker", *args], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)
    except FileNotFoundError:
        return 127, "docker was not found on this computer"
    return r.returncode, (r.stdout + r.stderr).strip()

# --- 4. Is the Day 19 stack running?
code, out = dc("compose", "ps", "--format", "{{.Service}} {{.State}}")
if code != 0:
    print("⚠️  Docker is not running or not installed:", out[:200])
    print("   Cells 2-5 still work. Start Docker Desktop (wait for the whale icon to stop moving) before the terminal step.")
else:
    print("Docker stack right now:", out.replace("\n", " | ") or "(nothing running; that is fine, the terminal step starts it)")
print(f"✅ ready. Python {sys.version.split()[0]} | folder: {os.getcwd()}")



Docker stack right now: (nothing running; that is fine, the terminal step starts it)
✅ ready. Python 3.11.14 | folder: c:\Users\rodne\Desktop\4 MONTHS JOURNEY\MONTH4-2


In [3]:
# The tools will test themselves(about 30 seconds; no Docker, no internet, no tokens)

# Runs the six Day 20 test files. Each starts a real little web server on this computer, sends real HTTP requests to it,
# and checks the logs, the /metrics text and the alert rules. No Docker, no internet and no tokens are used.
SUITES = ["test_monitoring.py", "test_bounded.py", "test_alerts.py", "test_log_report.py", "test_check_monitoring.py", "test_prepare.py"]
failed = []
for s in SUITES:
    t0 = time.time()
    r = subprocess.run([sys.executable, s], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=900)
    last = [l for l in r.stdout.splitlines() if " passed" in l][-1:] or ["(no result line)"]
    ok = r.returncode == 0
    print(("✅" if ok else "❌"), f"{s:<28}", last[0].strip(), f"  ({time.time() - t0:.0f}s)")
    if not ok:
        failed.append(s)
        print("   ", "\n    ".join(l for l in r.stdout.splitlines() if l.startswith("FAIL") or l.startswith("  File") or "Error" in l)[:1500])
        print("   ", r.stderr[-600:])
assert not failed, f"❌ failing: {failed}. Send me the lines above."
print("\n✅ all six test files pass on this computer")

✅ test_monitoring.py           37/37 passed   (9s)
✅ test_bounded.py              12/12 passed   (1s)
✅ test_alerts.py               17/17 passed   (10s)
✅ test_log_report.py           10/10 passed   (7s)
✅ test_check_monitoring.py     13/13 passed   (25s)
✅ test_prepare.py              12/12 passed   (6s)

✅ all six test files pass on this computer


In [4]:
# putting a ceiling on the lists that only grow

# Day 18's audit found lists/dicts that only ever grow. This puts a ceiling on each one (the oldest entries are dropped).
from audit_globals import audit
from make_bounded import make_bounded, remaining_growers
from freeze_kernel import verify_in_fresh_process

PIPE = "rag_pipeline/pipeline.py"
before = [f for f in audit(PIPE) if f["kind"] == "GROWS_FOREVER"]
print("grow forever BEFORE:", [f["name"] for f in before] or "nothing")

rep = make_bounded(PIPE)
print(rep.summary())
print("\ngrow forever AFTER :", remaining_growers(PIPE) or "nothing")

if rep.changed:
    t0 = time.time()
    problems = verify_in_fresh_process("rag_pipeline")
    print(f"(fresh process took {time.time() - t0:.0f}s)")
    print("\n".join("❌ " + p for p in problems) if problems else "✅ the package behaves exactly as before")
    assert not problems, "❌ the patched package no longer matches: the original is rag_pipeline/pipeline.py.before_bounded_patch"

grow forever BEFORE: ['VISUAL_ROUTE_LOG']
bounded  VISUAL_ROUTE_LOG       -> BoundedList (keeps the newest 1000)

grow forever AFTER : nothing
(fresh process took 132s)
✅ the package behaves exactly as before


In [5]:
# giving /metrics it's own key

# /metrics has its own key, different from the API keys, because the key goes to monitoring tools, not to users.
from prepare_monitoring import ensure_metrics_key
print(ensure_metrics_key(".env"))

gi = pathlib.Path(".gitignore")
lines = gi.read_text(encoding="utf-8").splitlines() if gi.exists() else []
add = [x for x in (".env.before_metrics_key", "*.before_bounded_patch") if x not in lines]
if add:
    gi.write_text("\n".join(lines + add) + "\n", encoding="utf-8")
    print("added to .gitignore:", ", ".join(add))

added RAG_METRICS_KEY to .env (a copy of the old file: .env.before_metrics_key). The value was not printed.
added to .gitignore: .env.before_metrics_key, *.before_bounded_patch


In [6]:
# Writes the Docker files again. The only change: the image now also contains monitoring.py.

import difflib
from container_check import check_container_readiness, fix_env_file
from make_requirements import write_requirements
from make_container_files import make_container_files
from prepare_monitoring import check_monitoring_ready

for line in fix_env_file(".env"):
    print("🔧", line)
check = check_container_readiness("rag_pipeline", ".")
assert check.ok, check.summary()
reqs = write_requirements("rag_pipeline", ".")
old = {n: pathlib.Path(n).read_text(encoding="utf-8").splitlines() for n in ("Dockerfile", ".dockerignore", "docker-compose.yml")}
written = make_container_files("rag_pipeline", ".", check, reqs)
print("wrote:", ", ".join(written))
for n, before in old.items():
    after = pathlib.Path(n).read_text(encoding="utf-8").splitlines()
    changes = [l for l in difflib.unified_diff(before, after, lineterm="", n=0) if l[:1] in "+-" and l[:3] not in ("+++", "---")]
    print(f"{n}: " + ("unchanged" if not changes else f"{len(changes)} changed line(s)"))
    for l in changes:
        print("    ", l)
code, out = dc("compose", "config", "-q")
print("docker-compose.yml:", "✅ valid" if code == 0 else "❌ " + out[-800:])

print()
ready = check_monitoring_ready(".", "rag_pipeline", ".env")
print(ready.summary())
assert ready.ok

wrote: Dockerfile, .dockerignore, docker-compose.yml, docker_healthcheck.py
Dockerfile: 4 changed line(s)
     -COPY --chown=app:app serve.py api_layer.py docker_healthcheck.py ./
     -#COPY --chown=app:app  ./
     +COPY --chown=app:app serve.py api_layer.py docker_healthcheck.py monitoring.py ./
     +COPY --chown=app:app  ./
.dockerignore: 2 changed line(s)
     +!monitoring.py
     +!
docker-compose.yml: 1 changed line(s)
     -    
docker-compose.yml: ✅ valid

✅ all Day 20 files are in the project folder
✅ serve.py is the Day 20 version
✅ RAG_METRICS_KEY is in .env and differs from your API keys
✅ nothing in the pipeline grows forever
✅ no logging call in the pipeline mentions question/answer-like names
✅ 3 print() call(s) mention question/answer-like names: print() is silenced inside the container (RAG_PIPELINE_PRINTS=1 would show them)
✅ the Dockerfile copies monitoring.py
✅ the port is published to this computer only (127.0.0.1), so /metrics is not open to the network

✅ ready

### Running .............
```
docker compose build
docker compose up -d
```

In [7]:
# The proof with one real question ( a few thousand tokens)

# Run AFTER the terminal step (docker compose build, docker compose up -d).
# Asks ONE real question (a few thousand tokens) and proves what monitoring recorded about it, and that nothing private leaked.
import httpx
from check_monitoring import check_monitoring, keys_from_env

K = keys_from_env(".env")
BASE = K["base_url"]

def wait_ready(max_seconds=900):
    """Poll /readyz until it says 200. After a rebuild the models come from the saved volume, so it is quicker than Day 19."""
    t0, last = time.time(), "nothing yet"
    while time.time() - t0 < max_seconds:
        try:
            r = httpx.get(f"{BASE}/readyz", timeout=5)
            if r.status_code == 200:
                return time.time() - t0
            last = f"readyz {r.status_code} {r.text[:200]}"
        except Exception as e:
            last = type(e).__name__
        time.sleep(3)
    raise AssertionError(f"❌ not ready after {max_seconds}s (last: {last}). Read the server's log:  docker compose logs --tail 80 rag")

print(f"ready after {wait_ready():.0f}s")
rep = check_monitoring(BASE, K["api_key"], K["metrics_key"], K["secrets"])
print(rep.summary())
assert rep.ok, "❌ read the ❌ lines above"


ready after 4s
✅ service answers /healthz (HTTP 200)
✅ /metrics refuses a request with no key (HTTP 401)
✅ /metrics refuses a wrong key (HTTP 401)
✅ /metrics refuses an API key (the two keys are separate) (HTTP 401)
✅ /metrics opens with the metrics key (HTTP 200)
✅ a real question was answered in 51.0s (HTTP 200)
✅ the answer carries a request id that matches the X-Request-ID header
✅ the /ask counter went up by exactly 1 (was 1)
✅ tokens counted for this question: 5,283
✅ estimated cost counted: $0.01839
✅ the answer was counted by kind (classification / retrieval source)
✅ the log has one line for this request (id dee8d58b...)
✅   it says: /ask, status 200, 51100 ms
✅   it names the user (rodney-api) and a hashed session (14f123c73b), not the raw session id
✅   pipeline time 51017 ms
✅   tokens in/out 4592/691, cost $0.01839
✅   kind of question: SIMPLE / local
✅ a request with a wrong key is refused (HTTP 401)
✅ nothing private in 32 log lines or on /metrics (question, answer, sess

In [8]:
# The numebrs and the story

# What the running service shows you. Two views: the numbers (/metrics) and the story (the log).
from log_report import compose_logs, parse, summarize
from alert_rules import fetch

snap = fetch(BASE, K["metrics_key"])
def show(label, value, fmt="{:,.0f}"):
    print(f"  {label:<44}", "-" if value is None else fmt.format(value))

print("=== right now (from /metrics) ===")
show("questions answered (HTTP 200 on /ask)", snap.get("rag_http_requests_total", route="/ask", status="200"))
show("tokens used, prompt", snap.get("rag_llm_tokens_total", kind="prompt"))
show("tokens used, completion", snap.get("rag_llm_tokens_total", kind="completion"))
show("estimated spend (USD)", snap.get("rag_llm_cost_usd_total"), "{:.4f}")
show("pipeline calls running now / slots", snap.get("rag_pipeline_in_flight"), "{:.0f}")
show("slots", snap.get("rag_pipeline_slots"))
show("memory held by the API process (MB)", (snap.get("rag_process_resident_memory_bytes") or 0) / 1024**2)
show("disk free here (GB)", (snap.get("rag_disk_free_bytes", path=".") or 0) / 1024**3, "{:.1f}")
for labels, v in snap.each("rag_dependency_up"):
    print(f"  dependency {labels['name']:<33}", "answering" if v == 1 else "NOT ANSWERING")
for labels, v in snap.each("rag_bounded_items"):
    print(f"  bounded list {labels['name']:<31}", f"{v:.0f} entries")

print("\n=== the story (from the log) ===")
events, other = parse(compose_logs("rag", 3000))
print(summarize(events, other))


=== right now (from /metrics) ===
  questions answered (HTTP 200 on /ask)        1
  tokens used, prompt                          4,592
  tokens used, completion                      691
  estimated spend (USD)                        0.0184
  pipeline calls running now / slots           0
  slots                                        2
  memory held by the API process (MB)          2,595
  disk free here (GB)                          939.5
  dependency mongo                             answering
  dependency neo4j                             answering
  dependency redis                             answering
  bounded list VISUAL_ROUTE_LOG                1 entries

=== the story (from the log) ===
LOG REPORT   2026-10-21 07:54:00 to 2026-10-21 07:57:25 UTC
32 lines: 21 structured events, 11 other lines

REQUESTS LOGGED: 5  (successful health/metrics checks are not logged)
  /metrics     401 x3
  /ask         200 x1   401 x1

ANSWERED QUESTIONS: 1 of 2
  time to answer   typical 51.1s  

In [14]:
from alert_rules import fetch, evaluate
current_metrics = fetch(BASE, K["metrics_key"])
active_alerts = evaluate(None, current_metrics)
print("Active alerts:", active_alerts)

Active alerts: []


In [15]:
# The fire drill
# A fire drill: break something on purpose and watch the alerts catch it, then repair it.
# Redis (conversation memory and rate limits) is stopped for about a minute. Two questions are sent while it is down.

from alert_rules import fetch, evaluate, Snapshot
import alert_rules as ar

before = fetch(BASE, K["metrics_key"])
assert not evaluate(None, before), "❌ something is already firing before the drill: fix that first (print evaluate(None, before))"
print("1. everything is healthy: no alerts firing")

print("2. stopping Redis ...")
code, out = dc("compose", "stop", "redis")
assert code == 0, out
time.sleep(12)                                              # /metrics re-checks the dependencies at most every 10 seconds
for i in range(2):
    r = httpx.post(f"{BASE}/ask", json={"question": f"drill question {i}", "session_id": "day20-drill"},
                   headers={"X-API-Key": K["api_key"]}, timeout=300)
    print(f"   a question while Redis is down -> HTTP {r.status_code} ({r.json().get('error', {}).get('code')})")
during = fetch(BASE, K["metrics_key"])
alerts = evaluate(before, during)
for a in alerts:
    print(f"   FIRING   [{a.severity}] {a.name}: {a.message}")
assert any(a.name == "dependency_down" and "redis" in a.message for a in alerts), "❌ the alert for a stopped Redis did not fire"
r = httpx.get(f"{BASE}/readyz")
print(f"   /readyz now says HTTP {r.status_code}: {r.json()['checks']}")

print("3. starting Redis again ...")
code, out = dc("compose", "start", "redis")
assert code == 0, out
t0 = time.time()
while time.time() - t0 < 120:
    time.sleep(3)
    if httpx.get(f"{BASE}/readyz").status_code == 200:
        break
time.sleep(11)
after = fetch(BASE, K["metrics_key"])
left = [a.name for a in evaluate(None, after)]
print(f"   {time.time() - t0:.0f}s later: alerts firing = {left or 'none'}")
assert not left, "❌ still firing after Redis came back"

print("\n4. what the log says about the two failed questions (error CLASS and place only, never the message):")
events, other = parse(compose_logs("rag", 400))
for e in [e for e in events if e.get("event") == "request" and e.get("route") == "/ask" and e.get("status", 0) >= 500][-2:]:
    print(f"   {e['ts']}  {e['status']}  {e.get('error')} at {e.get('error_at')}   request {e['rid'][:8]}...")
print("\n✅ the drill worked: the alert fired while Redis was down and cleared when it came back")

1. everything is healthy: no alerts firing
2. stopping Redis ...
   a question while Redis is down -> HTTP 504 (timeout)
   a question while Redis is down -> HTTP 504 (timeout)
   FIRING   [critical] dependency_down: redis is not answering
   FIRING   [warning] timeouts: 2 question(s) timed out (504)
   FIRING   [warning] abandoned_calls: 2 call(s) kept running after the caller was told it timed out
   /readyz now says HTTP 503: {'redis': 'fail: timeout', 'mongo': 'ok', 'neo4j': 'ok'}
3. starting Redis again ...
   21s later: alerts firing = none

4. what the log says about the two failed questions (error CLASS and place only, never the message):
   2026-10-21T09:56:07.737Z  504  None at None   request 7f048d1e...
   2026-10-21T09:58:10.676Z  504  None at None   request 9c9386a5...

✅ the drill worked: the alert fired while Redis was down and cleared when it came back


In [16]:
# When I finish for the day, I  stop everything and my data is kept.
code, out = dc("compose", "down")
print(out or "stopped")
print("Kept: the volumes models, checkpoints and redis_data. Start again any time with:   docker compose up -d")

Container rag-api-rag-1 Stopping 
 Container rag-api-rag-1 Stopped 
 Container rag-api-rag-1 Removing 
 Container rag-api-rag-1 Removed 
 Container rag-api-redis-1 Stopping 
 Container rag-api-redis-1 Stopped 
 Container rag-api-redis-1 Removing 
 Container rag-api-redis-1 Removed 
 Network rag-api_default Removing 
 Network rag-api_default Removed
Kept: the volumes models, checkpoints and redis_data. Start again any time with:   docker compose up -d
